In [ ]:
from project_utils import load_table, input_path, output_path, calculateCI, mean_error, standard_error, percentile_column, positive_fraction, SEED
import numpy as np
np.random.seed(SEED)


In [ ]:
# author Alex J. Yang
import numpy as np
import pandas as pd
import itertools


In [ ]:
import scipy
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm


In [ ]:
df = load_table('papers')


In [ ]:
dfNobel = load_table('nobel_papers')


In [ ]:
df['NobelField'] = df['PaperID'].map(dfNobel.set_index('paperid')['Field'])


In [ ]:
df['NobelField'] = df['NobelField'].fillna(0 )


In [ ]:
df[df['NobelField']!=0]


In [ ]:
df[df['NobelField']=='Medicine']


In [ ]:
# calculate CI using bootstrap


In [ ]:
def yearGroup(x):
    if x<1920:
        return '1900-1919'
    elif x<1940:
        return '1920-1939'
    elif x<1960:
        return '1940-1959'
    elif x<1980:
        return '1960-1979'
    else:
        return '1980-1999'
    
def delayGroup(x):
    if x<2:
        return x
    elif x<10:
        return '2-9'
    elif x<20:
        return '10-19'
    elif x<50:
        return '20-49'
    else:
        return '50+'
df['yearGroup'] = df['Year'].apply(yearGroup)
df['delayGroup'] = df['SB_T'].apply(delayGroup)


In [ ]:
df1= df[(df['type_Concave'] == 0)&(df['type_Immediacy'] == 0)].copy()
df2 = df[df['type_Concave'] == 1].copy()
df3 = df[df['type_Immediacy'] == 1].copy()


In [ ]:
convert_column_to_percentile = percentile_column
convert_column_to_percentile(df1, 'SB_B')


In [ ]:
def BpGroup(x):
    if pd.isna(x):
        return np.nan
    if x<5:
        return 5
    elif x<50:
        return 50
    elif x<95:
        return 95
    else:
        return 100
df1['SB_B_p'] = df1['SB_B_percentile'].apply(BpGroup)


In [ ]:
df.columns


In [ ]:
dflst = [df , df1 , df2 , df3]


# Hit papers


In [ ]:
x = range(6)
typelst = [0,1,'2-9', '10-19', '20-49','50+']
colum = 'Hit_1pct'
y = np.array([positive_fraction(df.loc[df['delayGroup'].eq(x), colum]) for x in typelst])

f, ax = plt.subplots(figsize=(9, 7))
plt.bar(x , y , capsize = 15 ,color='#424949', alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=20,)
plt.ylim(0,1)
plt.ylabel('P (Hit papers)',fontsize = 20)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
for _ in range(len(y)):
    plt.text(x[_], y[_]-0.04 , str(y[_].round(2)),fontsize = 16,c = 'white', ha='center', fontweight='bold')


In [ ]:
dff1 = df[df['Hit_1pct']==1]
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(dff1['SB_T'], dff1['Citation_Count'], c = 'red',marker='*',s=dff1['Citation_Count']*0.1, alpha = 0.01,label='Hit')
plt.ylim(100,1000000)
ax.set_yscale("log")
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.ylabel('Total Citation',fontsize = 30)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
plt.legend(fontsize = 30)
plt.show()


In [ ]:
x = range(6)
typelst = ['convex','immediacy','0-5','5-50','50-95','95-100']
CClst = [5,50,95,100]
colum = 'Hit_1pct'
y1 = np.array([ positive_fraction(df1.loc[df1['SB_B_p'].eq(x), colum]) for x in CClst])

y2 =  np.array(positive_fraction(df2[colum]))
y3 =  np.array(positive_fraction(df3[colum]))

y =  np.append(np.array([y2,y3]),y1)


In [ ]:
f, ax = plt.subplots(figsize=(10, 7))
plt.bar(x , y , capsize = 15 ,color=['#117A65','#A93226']+['#2471A3']*4, alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=15,)
plt.ylim(0,1)
plt.ylabel('P (Hit papers)',fontsize = 20)
plt.xlabel('B (percentile)',fontsize = 20)
for _ in range(len(y)):
    plt.text(x[_], y[_]-0.04 , str(y[_].round(2)),fontsize = 16,c = 'white', ha='center', fontweight='bold')


In [ ]:
dff1 = df1[df1['Hit_1pct']==1]


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(dff1['SB_B_percentile'], dff1['Citation_Count'], c = 'red',marker='*',s=dff1['Citation_Count']*0.1, alpha = 0.01,label='Hit')
plt.ylim(100,1000000)
ax.set_yscale("log")
plt.xlim(0,100)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.legend(fontsize = 30)
plt.ylabel('Total Citation',fontsize = 30)
plt.xlabel('B (percentile)',fontsize = 30)
plt.show()


# Nobel papers


In [ ]:
x = range(6)
typelst = [0,1,'2-9', '10-19', '20-49','50+']
colum = 'NobelField'
y = np.array([len(df.loc[(df['delayGroup']==x)&(df[colum]=='Physics') , colum]) for x in typelst])

f, ax = plt.subplots(figsize=(9, 7))
plt.bar(x , y , capsize = 15 ,color='#424949', alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=20,)
plt.ylim(0,150)
plt.ylabel('#Nobel papers (Physics)',fontsize = 20)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
for _ in range(len(y)):
    plt.text(x[_], y[_]-5 , str(int(float(y[_]))),fontsize = 16,c = 'white', ha='center', fontweight='bold')


In [ ]:
dff1 = df1[df1[colum]=='Physics']
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(dff1['SB_T'], dff1['Citation_Count'], c = '#922B21',marker='*',s=300, alpha = 0.9,label='Nobel')
plt.ylim(1,100000)
ax.set_yscale("log")
plt.xlim(-1,102)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.ylabel('Total Citation',fontsize = 30)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
plt.legend(fontsize = 30)
plt.show()


In [ ]:
x = range(6)
typelst = ['convex','immediacy','0-5','5-50','50-95','95-100']
CClst = [5,50,95,100]
colum = 'NobelField'
y1 = np.array([ len(df1.loc[(df1['SB_B_p']==x)&(df1[colum]=='Physics') , colum]) for x in CClst])

y2 =  np.array(len(df2.loc[df2[colum]=='Physics' , colum]))
y3 =  np.array(len(df3.loc[df3[colum]=='Physics' , colum]))

y =  np.append(np.array([y2,y3]),y1)


In [ ]:
f, ax = plt.subplots(figsize=(10, 7))
plt.bar(x , y , capsize = 15 ,color=['#117A65','#A93226']+['#2471A3']*4, alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=15,)
plt.ylim(0,150)
plt.ylabel('#Nobel papers (Physics)',fontsize = 20)
plt.xlabel('B (percentile)',fontsize = 20)
for _ in range(len(y)):
    if y[_]>10:
        plt.text(x[_], y[_]-5 , str(y[_].round(3)),fontsize = 16,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x[_], y[_]+1 , str(y[_].round(3)),fontsize = 16,c = 'black', ha='center')


In [ ]:
dff1 = df1[df1[colum]=='Physics']


In [ ]:
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(dff1['SB_B_percentile'], dff1['Citation_Count'], c = '#922B21',marker='*',s=300, alpha = 0.9,label='Nobel')
plt.ylim(1,100000)
ax.set_yscale("log")
plt.xlim(-1,102)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.ylabel('Total Citation',fontsize = 30)
plt.xlabel('B (percentile)',fontsize = 30)
plt.legend(fontsize = 30)
plt.show()


# Chemistry


In [ ]:
x = range(6)
typelst = [0,1,'2-9', '10-19', '20-49','50+']
colum = 'NobelField'
y = np.array([len(df.loc[(df['delayGroup']==x)&(df[colum]=='Chemistry') , colum]) for x in typelst])

f, ax = plt.subplots(figsize=(9, 7))
plt.bar(x , y , capsize = 15 ,color='#424949', alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=20,)
plt.ylim(0,150)
plt.ylabel('#Nobel papers (Chemistry)',fontsize = 20)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
for _ in range(len(y)):
    if y[_]>10:
        plt.text(x[_], y[_]-5 , str(y[_].round(3)),fontsize = 16,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x[_], y[_]+1 , str(y[_].round(3)),fontsize = 16,c = 'black', ha='center')


In [ ]:
dff1 = df1[df1[colum]=='Chemistry']
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(dff1['SB_T'], dff1['Citation_Count'], c = '#922B21',marker='*',s=300, alpha = 0.9,label='Nobel')
plt.ylim(1,100000)
ax.set_yscale("log")
plt.xlim(-1,102)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.ylabel('Total Citation',fontsize = 20)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
plt.legend(fontsize = 30)
plt.show()


In [ ]:
x = range(6)
typelst = ['convex','immediacy','0-5','5-50','50-95','95-100']
CClst = [5,50,95,100]
colum = 'NobelField'
y1 = np.array([ len(df1.loc[(df1['SB_B_p']==x)&(df1[colum]=='Chemistry') , colum]) for x in CClst])

y2 =  np.array(len(df2.loc[df2[colum]=='Chemistry' , colum]))
y3 =  np.array(len(df3.loc[df3[colum]=='Chemistry' , colum]))

y =  np.append(np.array([y2,y3]),y1)


In [ ]:
f, ax = plt.subplots(figsize=(10, 7))
plt.bar(x , y , capsize = 15 ,color=['#117A65','#A93226']+['#2471A3']*4, alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=15,)
plt.ylim(0,150)
plt.ylabel('#Nobel papers (Chemistry)',fontsize = 20)
plt.xlabel('B (percentile)',fontsize = 20)
for _ in range(len(y)):
    if y[_]>10:
        plt.text(x[_], y[_]-5 , str(y[_].round(3)),fontsize = 16,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x[_], y[_]+1 , str(y[_].round(3)),fontsize = 16,c = 'black', ha='center')


In [ ]:
dff1 = df1[df1[colum]=='Chemistry']


# Medicine


In [ ]:
x = range(6)
typelst = [0,1,'2-9', '10-19', '20-49','50+']
colum = 'NobelField'
y = np.array([len(df.loc[(df['delayGroup']==x)&(df[colum]=='Medicine') , colum]) for x in typelst])

f, ax = plt.subplots(figsize=(9, 7))
plt.bar(x , y , capsize = 15 ,color='#424949', alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=20,)
plt.ylim(0,150)
plt.ylabel('#Nobel papers (Medicine)',fontsize = 20)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
for _ in range(len(y)):
    plt.text(x[_], y[_]-5 , str(int(float(y[_]))),fontsize = 16,c = 'white', ha='center', fontweight='bold')


In [ ]:
dff1 = df1[df1[colum]=='Medicine']
f, ax = plt.subplots(figsize=(7, 7))
plt.scatter(dff1['SB_T'], dff1['Citation_Count'], c = '#922B21',marker='*',s=300, alpha = 0.9,label='Nobel')
plt.ylim(1,100000)
ax.set_yscale("log")
plt.xlim(-1,102)
plt.yticks(fontsize=20,)
plt.xticks(fontsize=20,)
plt.ylabel('Total Citation',fontsize = 30)
plt.xlabel(r'$𝑡_𝛼$',fontsize = 30)
plt.legend(fontsize = 30)
plt.show()


In [ ]:
x = range(6)
typelst = ['convex','immediacy','0-5','5-50','50-95','95-100']
CClst = [5,50,95,100]
colum = 'NobelField'
y1 = np.array([ len(df1.loc[(df1['SB_B_p']==x)&(df1[colum]=='Medicine') , colum]) for x in CClst])

y2 =  np.array(len(df2.loc[df2[colum]=='Medicine' , colum]))
y3 =  np.array(len(df3.loc[df3[colum]=='Medicine' , colum]))

y =  np.append(np.array([y2,y3]),y1)


In [ ]:
f, ax = plt.subplots(figsize=(10, 7))
plt.bar(x , y , capsize = 15 ,color=['#117A65','#A93226']+['#2471A3']*4, alpha = .9)
plt.yticks(fontsize=20,)
plt.xticks(x,typelst,fontsize=15,)
plt.ylim(0,150)
plt.ylabel('#Nobel papers (Medicine)',fontsize = 20)
plt.xlabel('B (percentile)',fontsize = 20)
for _ in range(len(y)):
    if y[_]>10:
        plt.text(x[_], y[_]-5 , str(y[_].round(3)),fontsize = 16,c = 'white', ha='center', fontweight='bold')
    else:
        plt.text(x[_], y[_]+1 , str(y[_].round(3)),fontsize = 16,c = 'black', ha='center')


In [ ]:
dff1 = df1[df1[colum]=='Medicine']
